# nn-parameter-wrap — worked example 2: Parameter vs buffer in a LayerNorm-style module

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `nn-parameter-wrap`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

A learnable scale belongs in `nn.Parameter` (trained by the optimizer); a non-learnable running statistic belongs in a buffer registered with `register_buffer`. Both appear in `state_dict`, but only the Parameter appears in `.parameters()`.

## Worked solution

We build `RunningNorm`, a stripped LayerNorm-like module. After `super().__init__()` we register a learnable `gamma` as `nn.Parameter(t.ones(dim))` and a non-learnable `running_var` as a buffer initialized to ones via `register_buffer`. In `forward` during training we update `running_var` toward the batch variance with an EMA, wrapped in `t.no_grad()` so the update never enters the autograd graph; then we scale the input by `gamma`. The key checks: `gamma` is in `.parameters()`, `running_var` is in `.buffers()` but NOT in `.parameters()`, and both round-trip through `state_dict()`. We print the parameter and buffer names to confirm the split.

In [ ]:
import torch as t
import torch.nn as nn

t.manual_seed(0)

class RunningNorm(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.gamma = nn.Parameter(t.ones(dim))
        self.register_buffer('running_var', t.ones(dim))
    def forward(self, x):
        if self.training:
            with t.no_grad():
                self.running_var = 0.9 * self.running_var + 0.1 * x.var(dim=0, unbiased=False)
        return x * self.gamma

mod = RunningNorm(4)
print('param names:', [n for n, _ in mod.named_parameters()])
print('buffer names:', [n for n, _ in mod.named_buffers()])
print('state_dict keys:', list(mod.state_dict().keys()))